# Metorial + Microsoft Autogen Example

This notebook demonstrates how to use Metorial tools with Microsoft Autogen multi-agent conversations.

In [ ]:
# Install dependencies
%pip install metorial pyautogen python-dotenv

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

assert os.getenv("METORIAL_API_KEY"), "Set METORIAL_API_KEY"
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY"
assert os.getenv("EXA_DEPLOYMENT_ID"), "Set EXA_DEPLOYMENT_ID"

In [ ]:
from autogen import AssistantAgent, UserProxyAgent

from metorial import Metorial
from metorial.integrations.autogen import (
  create_autogen_tools,
  get_autogen_tool_executor,
)

In [ ]:
metorial = Metorial(api_key=os.getenv("METORIAL_API_KEY"))

In [ ]:
async def run_agents(query: str):
  async with metorial.provider_session(
    provider="openai",
    server_deployments=[os.getenv("EXA_DEPLOYMENT_ID")],
  ) as session:
    tools = create_autogen_tools(session)
    tool_executor = get_autogen_tool_executor(session)

    print(f"Available tools: {[t['function']['name'] for t in tools]}")

    llm_config = {
      "config_list": [{"model": "gpt-4o", "api_key": os.getenv("OPENAI_API_KEY")}],
      "tools": tools,
    }

    assistant = AssistantAgent(
      name="research_assistant",
      system_message="You are a helpful research assistant.",
      llm_config=llm_config,
    )

    user_proxy = UserProxyAgent(
      name="user",
      human_input_mode="NEVER",
      max_consecutive_auto_reply=5,
      function_map=tool_executor,
    )

    await user_proxy.a_initiate_chat(assistant, message=query)

In [ ]:
await run_agents("Search for the latest Python 3.13 features")